# Dataset Person Name Train

In [6]:
import pandas as pd
import json
import spacy
from spacy.tokens import DocBin


## Pilih Dataset

In [7]:
# Baca CSV menggunakan Pandas.
# Sesuaikan nomor versi dengan file yang
# benar-benar dihasilkan generator run terakhir (sekarang berisi PER + ADR).
df = pd.read_csv("dataset_ner_3000_v1.csv")

# Split train/dev supaya evaluasi mencerminkan generalisasi, bukan hafalan
# data yang sama persis dengan yang dipakai untuk training.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # acak urutan
split_idx = int(len(df) * 0.9)
df_train = df.iloc[:split_idx]
df_dev = df.iloc[split_idx:]
print(f"Total data: {len(df)} | Train: {len(df_train)} | Dev: {len(df_dev)}")

Total data: 3000 | Train: 2700 | Dev: 300


## Ubah Dataset Ke Format .spacy

In [8]:
nlp = spacy.blank("xx")


def buat_doc_bin(dataframe):
    doc_bin = DocBin()
    for _, row in dataframe.iterrows():
        teks = row["teks"]
        # Parse kembali string JSON dari CSV menjadi list Python
        entitas = json.loads(row["entitas"])

        doc = nlp.make_doc(teks)
        ents = []

        for ent in entitas:
            span = doc.char_span(
                ent["start"], ent["end"], label=ent["label"], alignment_mode="contract"
            )
            if span is not None:
                ents.append(span)

        doc.ents = ents
        doc_bin.add(doc)
    return doc_bin


buat_doc_bin(df_train).to_disk("./train.spacy")
buat_doc_bin(df_dev).to_disk("./dev.spacy")
print("Data CSV berhasil dikonversi ke train.spacy dan dev.spacy!")

Data CSV berhasil dikonversi ke train.spacy dan dev.spacy!


## Konfigurasi Fine-Tuning (Sourced dari Model Pretrained)

`config.cfg` **sudah ditulis manual**, bukan hasil `spacy init config`.
Komponen `ner` di-*source* langsung dari pipeline resmi spaCy
[`xx_ent_wiki_sm`](https://github.com/explosion/spacy-models) (multi-bahasa,
dilatih dari data WikiNER, terpasang lewat `uv add`), lengkap dengan bobotnya:

```
[components.ner]
source = "xx_ent_wiki_sm"
```

Karena tidak dimasukkan ke `frozen_components`, bobot ini tetap ikut
di-update (fine-tune) saat training di bawah — bukan training dari 0.

**Dataset sekarang gabungan 2 label: `PER` dan `ADR`.**
`xx_ent_wiki_sm` punya `lang = "xx"` dan label asli `PER/ORG/LOC/MISC`.
Label `PER` cocok langsung (warm-start, bukan dari 0). Label `ADR`
**tidak ada** di skema aslinya, jadi spaCy akan otomatis memanggil
`add_label("ADR")` saat inisialisasi training - ini menambah slot output
baru yang mulai dari bobot acak (tidak warm-start), tapi tetap memanfaatkan
representasi tok2vec yang sudah dilatih dari `xx_ent_wiki_sm`, jadi tetap
jauh lebih baik daripada training dari 0 sepenuhnya.

Trade-off yang sama seperti sebelumnya tetap berlaku: karena dataset kita
hanya melabeli PER dan ADR (token lain jadi "O"), kemampuan mengenali
ORG/LOC/MISC asli dari `xx_ent_wiki_sm` akan berangsur memudar setelah
fine-tuning selesai.

**JANGAN jalankan `spacy init config ... --force` lagi** di project ini -
itu akan menimpa baris `source = ...` di atas dan membuat training
kembali dari 0.

In [9]:
# Sengaja TIDAK menjalankan `spacy init config ... --force` di sini lagi.
# config.cfg sudah ditulis manual untuk fine-tuning (sourcing ner dari
# id_ner_spacy_indonesian) - lihat penjelasan di cell markdown di atas.
# Cukup lanjut ke cell training di bawah.
print("Skip: config.cfg sudah dikonfigurasi manual untuk fine-tuning.")

Skip: config.cfg sudah dikonfigurasi manual untuk fine-tuning.


## Run Proses Training 

In [10]:
!python -m spacy train config.cfg --output ./models --paths.train ./train.spacy --paths.dev ./dev.spacy

ℹ Saving to output directory: models
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['ner']
ℹ Initial learn rate: 0.001
E    #       LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  --------  ------  ------  ------  ------
  0       0     26.44   29.01   20.71   48.42    0.29
  0     200   1597.33   95.06   96.46   93.70    0.95
  1     400    212.09   97.99   97.99   97.99    0.98
  2     600     73.69   98.85   98.85   98.85    0.99
  3     800     40.37   98.43   98.01   98.85    0.98
  4    1000     25.10   99.71  100.00   99.43    1.00
  6    1200     18.31   99.28   99.14   99.43    0.99
  7    1400     23.12  100.00  100.00  100.00    1.00
 10    1600     14.64  100.00  100.00  100.00    1.00
 13    1800     19.72   99.71   99.71   99.71    1.00
 16    2000     22.17   99.43   99.43   99.43    0.99
 21    2200     21.0